In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по птице v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Птица
1944,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2023-11-01,591.27
1392,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2021-01-01,2.30
1668,ПАВЛОДАРСКАЯ ОБЛАСТЬ,2018-08-01,556.12
515,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2021-02-01,3640.86
762,ГШЫМКЕНТ,2021-05-01,189.40
726,ГАСТАНА,2025-05-01,0.00
1232,КОСТАНАЙСКАЯ ОБЛАСТЬ,2018-03-01,568.83
1468,МАНГИСТАУСКАЯ ОБЛАСТЬ,2021-01-01,36.60
1914,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2021-05-01,1021.63
914,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2023-06-01,1616.56


In [3]:
regions = df['Регион'].unique()
target   = "Птица"
horizon  = 3
epsilon = 1e-6

In [4]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [5]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Птица - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Птица - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[11268.8, 10962.08, 10960.74]","[10080.07, 10273.59, 10251.41]",892.61,862.18,8.47
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[10252.48, 10257.31, 10525.67]","[10273.59, 10251.41, 8665.21]",1074.21,629.16,7.24
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[10269.94, 10538.61, 11740.42]","[10251.41, 8665.21, 11438.74]",1095.59,731.20,8.15
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[10527.16, 11727.89, 9571.69]","[8665.21, 11438.74, 8560.3]",1234.69,1054.16,11.94
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[10434.02, 8525.1, 8451.39]","[11438.74, 8560.3, 12551.51]",2437.33,1713.35,13.95
...,...,...,...,...,...,...,...,...
168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[354.92, 526.86, 541.42]","[810.6, 641.64, 802.3]",310.31,277.11,35.54
169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[751.03, 767.97, 1103.05]","[641.64, 802.3, 750.07]",214.27,165.57,22.80
170,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[717.59, 1031.78, 758.58]","[802.3, 750.07, 754.31]",169.86,123.56,16.23
171,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[1081.73, 794.83, 942.02]","[750.07, 754.31, 925.67]",193.14,129.51,17.12


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,1670.51,1445.79,13.02
1,АКТЮБИНСКАЯ ОБЛАСТЬ,34.03,26.98,277.84
2,АЛМАТИНСКАЯ ОБЛАСТЬ,1183.86,1044.20,8.81
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,569.14,522.48,8.55
4,ГАЛМАТЫ,0.11,0.11,NaN
5,ГАСТАНА,0.08,0.07,NaN
6,ГШЫМКЕНТ,8.77,7.97,168.34
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,584.18,511.77,25.54
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,265.84,214.58,20.69
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,96.68,80.45,8.14


## SARIMA

In [6]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Птица - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Птица - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(3, 1, 0)","(0, 1, 1, 12)",1560.86,1520.20,14.93,"[12055.74, 11745.79, 11364.15]","[10080.07, 10273.59, 10251.41]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(2, 1, 3)","(1, 1, 1, 12)",912.89,666.89,7.42,"[10421.65, 9942.4, 10208.81]","[10273.59, 10251.41, 8665.21]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(3, 1, 2)","(0, 1, 1, 12)",979.81,715.54,7.93,"[9787.87, 10296.94, 11490.09]","[10251.41, 8665.21, 11438.74]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(3, 1, 2)","(0, 1, 1, 12)",1506.50,1383.30,15.44,"[10758.13, 12071.8, 9984.22]","[8665.21, 11438.74, 8560.3]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(3, 1, 3)","(1, 1, 1, 12)",1195.23,831.65,6.88,"[11120.95, 8696.32, 10510.38]","[11438.74, 8560.3, 12551.51]"
...,...,...,...,...,...,...,...,...,...,...
171,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(2, 0, 0)","(2, 1, 0, 12)",424.45,419.92,55.68,"[318.42, 300.59, 375.77]","[810.6, 641.64, 802.3]"
172,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(2, 0, 0)","(2, 1, 0, 12)",152.33,146.31,20.62,"[459.65, 715.55, 920.27]","[641.64, 802.3, 750.07]"
173,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(1, 0, 1)","(2, 1, 0, 12)",191.13,158.24,20.89,"[738.0, 1058.32, 652.14]","[802.3, 750.07, 754.31]"
174,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(1, 0, 1)","(2, 1, 0, 12)",276.34,249.14,30.52,"[1098.55, 673.31, 607.73]","[750.07, 754.31, 925.67]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,1567.44,1371.08,12.89
1,АКТЮБИНСКАЯ ОБЛАСТЬ,41.10,31.75,285.20
2,АЛМАТИНСКАЯ ОБЛАСТЬ,1419.89,1202.61,9.95
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,468.71,428.08,7.01
4,ГАЛМАТЫ,7.13,4.61,4612.45
5,ГАСТАНА,0.00,0.00,127572181825238466560.00
6,ГШЫМКЕНТ,8.19,6.90,131.51
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,604.21,552.49,28.21
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,178.20,151.70,14.34
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,161.51,152.15,14.98


## Facebook Prophet

In [7]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Птица - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Птица - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


10:45:24 - cmdstanpy - INFO - Chain [1] start processing
10:45:26 - cmdstanpy - INFO - Chain [1] done processing
10:45:26 - cmdstanpy - INFO - Chain [1] start processing
10:45:26 - cmdstanpy - INFO - Chain [1] done processing
10:45:26 - cmdstanpy - INFO - Chain [1] start processing
10:45:27 - cmdstanpy - INFO - Chain [1] done processing
10:45:27 - cmdstanpy - INFO - Chain [1] start processing
10:45:27 - cmdstanpy - INFO - Chain [1] done processing
10:45:27 - cmdstanpy - INFO - Chain [1] start processing
10:45:27 - cmdstanpy - INFO - Chain [1] done processing
10:45:28 - cmdstanpy - INFO - Chain [1] start processing
10:45:28 - cmdstanpy - INFO - Chain [1] done processing
10:45:28 - cmdstanpy - INFO - Chain [1] start processing
10:45:28 - cmdstanpy - INFO - Chain [1] done processing
10:45:29 - cmdstanpy - INFO - Chain [1] start processing
10:45:29 - cmdstanpy - INFO - Chain [1] done processing
10:45:29 - cmdstanpy - INFO - Chain [1] start processing
10:45:29 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,1137.18,1052.46,10.33,"[11449.17, 11618.36, 10694.92]","[10080.07, 10273.59, 10251.41]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,1775.23,1457.59,15.87,"[11516.36, 10589.34, 11457.28]","[10273.59, 10251.41, 8665.21]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,1862.42,1572.69,16.40,"[10549.87, 11399.51, 13124.06]","[10251.41, 8665.21, 11438.74]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,1954.87,1834.76,19.65,"[11390.14, 13125.83, 9652.56]","[8665.21, 11438.74, 8560.3]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,1638.35,1551.15,13.91,"[12915.87, 9505.75, 10320.64]","[11438.74, 8560.3, 12551.51]"
...,...,...,...,...,...,...,...,...
171,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,363.12,309.16,39.30,"[251.87, 545.02, 530.17]","[810.6, 641.64, 802.3]"
172,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,134.29,114.14,15.10,"[575.01, 588.16, 811.72]","[641.64, 802.3, 750.07]"
173,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,161.11,155.71,20.17,"[612.89, 847.56, 574.06]","[802.3, 750.07, 754.31]"
174,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,177.33,170.49,20.75,"[860.63, 583.47, 695.6]","[750.07, 754.31, 925.67]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,1545.90,1379.94,12.92
1,АКТЮБИНСКАЯ ОБЛАСТЬ,38.72,28.84,143.70
2,АЛМАТИНСКАЯ ОБЛАСТЬ,1330.38,1211.75,10.37
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,847.36,728.83,12.13
4,ГАЛМАТЫ,0.10,0.10,99.82
5,ГАСТАНА,0.00,0.00,NaN
6,ГШЫМКЕНТ,33.23,30.90,673.22
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,694.34,643.10,32.70
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,342.36,302.07,29.70
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,120.31,93.85,9.47


In [8]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Птица - Лучшие модели v2.xlsx", index=False)


                        Регион  MAPE_HW              MAPE_SARIMA  MAPE_Prophet Best_method
           АКМОЛИНСКАЯ ОБЛАСТЬ    13.02                    12.89         12.92      SARIMA
           АКТЮБИНСКАЯ ОБЛАСТЬ   277.84                   285.20        143.70     Prophet
           АЛМАТИНСКАЯ ОБЛАСТЬ     8.81                     9.95         10.37          HW
ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     8.55                     7.01         12.13      SARIMA
                       ГАЛМАТЫ      NaN                  4612.45         99.82     Prophet
                       ГАСТАНА      NaN 127572181825238466560.00           NaN      SARIMA
                      ГШЫМКЕНТ   168.34                   131.51        673.22      SARIMA
            ЖАМБЫЛСКАЯ ОБЛАСТЬ    25.54                    28.21         32.70          HW
 ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ    20.69                    14.34         29.70      SARIMA
        КАРАГАНДИНСКАЯ ОБЛАСТЬ     8.14                    14.98          9.47          HW

In [9]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Птица - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Птица - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Птица - Результаты прогнозов Prophet средние v2.xlsx"


# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [10]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Птица - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,13.02,1445.79,12.89,1371.08,12.92,1379.94,SARIMA,MAPE,12.89,1371.08
1,АКТЮБИНСКАЯ ОБЛАСТЬ,277.84,26.98,285.20,31.75,143.70,28.84,Prophet,MAPE,143.70,26.98
2,АЛМАТИНСКАЯ ОБЛАСТЬ,8.81,1044.20,9.95,1202.61,10.37,1211.75,HW,MAPE,8.81,1044.20
3,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,8.55,522.48,7.01,428.08,12.13,728.83,SARIMA,MAPE,7.01,428.08
4,ГАЛМАТЫ,NaN,0.11,4612.45,4.61,99.82,0.10,Prophet,MAPE,99.82,0.10
5,ГАСТАНА,NaN,0.07,127572181825238499328.00,0.00,NaN,0.00,SARIMA,MAE,127572181825238499328.00,0.00
6,ГШЫМКЕНТ,168.34,7.97,131.51,6.90,673.22,30.90,SARIMA,MAPE,131.51,6.90
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,25.54,511.77,28.21,552.49,32.70,643.10,HW,MAPE,25.54,511.77
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,20.69,214.58,14.34,151.70,29.70,302.07,SARIMA,MAPE,14.34,151.70
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,8.14,80.45,14.98,152.15,9.47,93.85,HW,MAPE,8.14,80.45
